In [1]:
from mcp_utils.simple_mcp_client import SimpleMCPClient

In [2]:
mcp_client = SimpleMCPClient()

In [3]:
mcp_client.connect("http://localhost:8080/sse")

In [4]:
import json

tools = mcp_client.get_tools()
print(
    "\n".join(
        [
            f"{tool.name}\n{tool.description}\n"
            + json.dumps(
                {k: v["description"] for k, v in tool.inputSchema["properties"].items()}
            )
            + "\n"
            for tool in tools.tools
        ]
    )
)

Human-Feedback Tool
A tool that allows interaction with a human. Use it when you want some clarifications or feedback on the task that you are performing, to ensure that you are doing the right thing.
{"query": "The query to ask the human."}

Python Interpreter Tool
Execute an arbitrary Python code passed as input. 
Important: this does not evaluate the code as an expression! For instance, `return` statements won't work.
Instead, in order to return a final result, you have to assign whatever you want to return to the `result` variable.
{"code": "The Python code to be executed."}

Final Answer Tool
This tool is required to complete your assigned task. When you are ready to provide a final answer to the user, make sure to call this tool to complete your task, passing your final answer as a parameter. Make sure to only call this tool once you are absolutely sure of the final result you want to submit; after calling this tool, you won't be able to continue and the process will be interrupt

In [5]:
response = mcp_client.call_tool(
    "Python Interpreter Tool",
    {"code": "result = 1\nfor i in range(1, 15):\n    result *= i"},
)
print(response.content[0].text)

87178291200


---

In [ ]:
from pydantic import BaseModel, Field


class Parameters(BaseModel):
    query: str = Field(..., description="The query to ask the human.")


parameters = Parameters(**{"query": "Who are you?"})
parameters


Parameters(query='Who are you?')

In [11]:
await mcp_client.call_tool("Human-Feedback Tool", parameters)

ValidationError: 1 validation error for CallToolRequestParams
arguments
  Input should be a valid dictionary [type=dict_type, input_value=Parameters(query='Who are you?'), input_type=Parameters]
    For further information visit https://errors.pydantic.dev/2.10/v/dict_type

In [ ]:
await mcp_client.cleanup()